In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
!nvidia-smi

Sat Aug  1 21:45:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.49                 Driver Version: 596.49         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P8              3W /   80W |     149MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# !pip install -q cupy-cuda12x

In [4]:
import cupy as cp

print("CuPy version:", cp.__version__)
print("CUDA devices:", cp.cuda.runtime.getDeviceCount())
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())

CuPy version: 14.1.1
CUDA devices: 1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
import cupy as cp

x = cp.arange(10)
print(x)
print(type(x))

[0 1 2 3 4 5 6 7 8 9]
<class 'cupy.ndarray'>


In [6]:
import numpy as np
import cupy as cp
import time

N = 4096

# NumPy
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

t0 = time.perf_counter()
C = A @ B
t1 = time.perf_counter()

print(f"NumPy: {t1 - t0:.3f} s")

# CuPy
A_gpu = cp.asarray(A)
B_gpu = cp.asarray(B)

# Warm up
_ = A_gpu @ B_gpu
cp.cuda.Stream.null.synchronize()

t0 = time.perf_counter()
C_gpu = A_gpu @ B_gpu
cp.cuda.Stream.null.synchronize()
t1 = time.perf_counter()

print(f"CuPy: {t1 - t0:.3f} s")

NumPy: 0.314 s
CuPy: 0.034 s


In [7]:
import numpy as np
import cupy as cp
import time

# ------------------------
# Network Configuration
# ------------------------
batch_size = 8192
input_dim = 2048
hidden_dim = 4096
output_dim = 2048
# ------------------------
# NumPy
# ------------------------
X = np.random.randn(batch_size, input_dim).astype(np.float32)

W1 = np.random.randn(input_dim, hidden_dim).astype(np.float32)
b1 = np.random.randn(hidden_dim).astype(np.float32)

W2 = np.random.randn(hidden_dim, output_dim).astype(np.float32)
b2 = np.random.randn(output_dim).astype(np.float32)

start = time.perf_counter()

H = X @ W1 + b1
H = np.maximum(H, 0)      # ReLU
Y = H @ W2 + b2

numpy_time = time.perf_counter() - start

print(f"NumPy Forward Pass: {numpy_time:.4f} seconds")

# ------------------------
# CuPy
# ------------------------
X_gpu = cp.asarray(X)
W1_gpu = cp.asarray(W1)
b1_gpu = cp.asarray(b1)

W2_gpu = cp.asarray(W2)
b2_gpu = cp.asarray(b2)

# Warm-up
_ = cp.maximum(X_gpu @ W1_gpu + b1_gpu, 0) @ W2_gpu + b2_gpu
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()

H_gpu = X_gpu @ W1_gpu + b1_gpu
H_gpu = cp.maximum(H_gpu, 0)
Y_gpu = H_gpu @ W2_gpu + b2_gpu

cp.cuda.Stream.null.synchronize()

cupy_time = time.perf_counter() - start

print(f"CuPy Forward Pass: {cupy_time:.4f} seconds")

print(f"\nSpeedup: {numpy_time / cupy_time:.2f}x")

NumPy Forward Pass: 0.7510 seconds
CuPy Forward Pass: 0.0580 seconds

Speedup: 12.94x


In [8]:
import sys

print(sys.executable)

x:\VS_CODE\LLM_track\.venv\Scripts\python.exe


In [10]:
import numpy as np
import cupy as cp
import time

# ============================
# Network Configuration
# ============================

batch_size = 8192
input_dim = 2048
hidden_dim = 4096
output_dim = 1024
num_hidden_layers = 8
iterations = 50

# ============================
# NumPy Network
# ============================

weights_np = []
biases_np = []

dims = [input_dim] + [hidden_dim] * num_hidden_layers + [output_dim]

for i in range(len(dims) - 1):
    W = np.random.randn(dims[i], dims[i + 1]).astype(np.float32) * 0.01
    b = np.random.randn(dims[i + 1]).astype(np.float32) * 0.01
    weights_np.append(W)
    biases_np.append(b)

X_np = np.random.randn(batch_size, input_dim).astype(np.float32)

# # Warm-up
# H = X_np
# for W, b in zip(weights_np, biases_np):
#     H = H @ W + b
#     H = np.maximum(H, 0)

# start = time.perf_counter()

# for _ in range(iterations):
#     H = X_np
#     for W, b in zip(weights_np, biases_np):
#         H = H @ W + b
#         H = np.maximum(H, 0)

# numpy_time = time.perf_counter() - start

# print(f"NumPy Time : {numpy_time:.3f} seconds")

# ============================
# CuPy Network
# ============================

weights_cp = [cp.asarray(W) for W in weights_np]
biases_cp = [cp.asarray(b) for b in biases_np]

X_cp = cp.asarray(X_np)

# Warm-up
H = X_cp
for W, b in zip(weights_cp, biases_cp):
    H = H @ W + b
    H = cp.maximum(H, 0)

cp.cuda.Stream.null.synchronize()

start = time.perf_counter()

for _ in range(iterations):
    H = X_cp
    for W, b in zip(weights_cp, biases_cp):
        H = H @ W + b
        H = cp.maximum(H, 0)

cp.cuda.Stream.null.synchronize()

cupy_time = time.perf_counter() - start

print(f"CuPy Time  : {cupy_time:.3f} seconds")

# print(f"\nGPU Speedup : {numpy_time / cupy_time:.2f}x")

CuPy Time  : 21.855 seconds
